# Mamba vs. Transformer for NLP on IMDb Dataset

This notebook aims to experimentally compare the Mamba architecture with the Transformer architecture on a Natural Language Processing (NLP) task, specifically sentiment analysis using the IMDb movie review dataset. We will explore fine-tuning pre-trained models for both architectures and analyze their performance across different sequence lengths.

## 1. Setup and Library Installation

First, we'll install the necessary libraries, including `transformers` for pre-trained Transformer models and utilities, `datasets` for easy data loading, `accelerate` for distributed training, `evaluate` for metrics, and `mamba-ssm` for the Mamba model.

In [1]:
!pip install --force-reinstall torch==2.5.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.0%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/n

In [1]:
import sys
import torch

# 1. Grab exact runtime metrics from Colab
# py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
py_ver = f"cp310"
torch_ver = "2.5"

print(f"Detected Python: {py_ver}")
print(f"Detected PyTorch: torch{torch_ver}")

# 2. Build explicit, direct wheel strings matching current assets
# (Using modern v1.6.2 and v2.3.2 precompiled paths)
base_conv = f"causal_conv1d-1.6.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"
base_mamba = f"mamba_ssm-2.3.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"

conv_url = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/{base_conv}"
mamba_url = f"https://github.com/state-spaces/mamba/releases/download/v2.3.0/{base_mamba}"

# 3. Pull prerequisites instantly
# !pip install packaging ninja --quiet

# 4. Force install straight from the exact URL strings
print(f"uv pip install {conv_url} --no-build-isolation")
print(f"uv pip install {mamba_url} --no-build-isolation")


Detected Python: cp310
Detected PyTorch: torch2.5
uv pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl --no-build-isolation
uv pip install https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl --no-build-isolation


In [2]:
import torch
try:
    import causal_conv1d
    import mamba_ssm
    print("Success: Mamba has been successfully unpacked and verified via binary wheels!")
except ImportError as e:
    print(f"Initialization failed: {e}")

/home/sikora/studia/sem6/llm/projekt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Success: Mamba has been successfully unpacked and verified via binary wheels!


In [6]:
!pip install transformers datasets accelerate evaluate

  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.4.0
    Uninstalling fsspec-2026.4.0:
      Successfully uninstalled fsspec-2026.4.0


In [3]:
# Targeted diagnostic
import pkgutil
import mamba_ssm.ops.triton as triton_pkg

print("=== mamba_ssm.ops.triton submodules ===")
print([m.name for m in pkgutil.iter_modules(triton_pkg.__path__)])

try:
    from mamba_ssm.ops.triton.selective_state_update import selective_state_update
    print("\nselective_state_update: FOUND in triton.selective_state_update")
except Exception as e:
    print(f"\nselective_state_update: NOT FOUND — {e}")

import causal_conv1d
print("\n=== causal_conv1d top-level ===")
print([x for x in dir(causal_conv1d) if not x.startswith("_")])

try:
    from causal_conv1d import causal_conv1d_update
    print("\ncausal_conv1d_update: FOUND")
except Exception as e:
    print(f"\ncausal_conv1d_update: NOT FOUND — {e}")

=== mamba_ssm.ops.triton submodules ===
['k_activations', 'layer_norm', 'layernorm_gated', 'selective_state_update', 'softplus', 'ssd_bmm', 'ssd_chunk_scan', 'ssd_chunk_state', 'ssd_combined', 'ssd_state_passing']

selective_state_update: FOUND in triton.selective_state_update

=== causal_conv1d top-level ===
['causal_conv1d_fn', 'causal_conv1d_interface', 'causal_conv1d_update', 'causal_conv1d_varlen', 'cpp_functions']

causal_conv1d_update: FOUND


In [4]:
import mamba_ssm
from mamba_ssm.ops.triton.selective_state_update import selective_state_update
from causal_conv1d import causal_conv1d_fn, causal_conv1d_update

# Already at top-level: selective_scan_fn, mamba_inner_fn
# Already in causal_conv1d: causal_conv1d_fn, causal_conv1d_update
# Only missing one — patch it in:
# if not hasattr(mamba_ssm, 'selective_state_update'):
mamba_ssm.selective_state_update = selective_state_update

for name, val in [
    ("selective_state_update", mamba_ssm.selective_state_update),
    ("selective_scan_fn",      mamba_ssm.selective_scan_fn),
    ("mamba_inner_fn",         mamba_ssm.mamba_inner_fn),
    ("causal_conv1d_fn",       causal_conv1d_fn),
    ("causal_conv1d_update",   causal_conv1d_update),
]:
    print(f"{name:30s} → {bool(val):5}  {type(val)}")

# Verify all 5 that transformers checks for the fast path
all_present = all([
    mamba_ssm.selective_state_update,
    mamba_ssm.selective_scan_fn,
    mamba_ssm.mamba_inner_fn,
    causal_conv1d_fn,
    causal_conv1d_update,
])
print("Fast path available:", all_present)  # should be True

selective_state_update         →     1  <class 'function'>
selective_scan_fn              →     1  <class 'function'>
mamba_inner_fn                 →     1  <class 'function'>
causal_conv1d_fn               →     1  <class 'function'>
causal_conv1d_update           →     1  <class 'function'>
Fast path available: True


In [5]:
import torch
import torch.nn as nn
import numpy as np
import evaluate
import datasets
from transformers import (
    AutoTokenizer, AutoConfig,
    MambaPreTrainedModel, MambaModel,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [6]:
def compute_metrics(eval_pred):
    acc_metric = evaluate.load("accuracy")
    f1_metric  = evaluate.load("f1")
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1":        f1_metric.compute(predictions=preds,  references=labels, average="weighted")["f1"],
    }

## 2. Dataset Loading and Preprocessing

We will load the `imdb` dataset from the Hugging Face `datasets` library, which contains movie reviews labeled as positive or negative sentiment. After loading, we will tokenize the text and prepare the data for model training.

In [7]:
# Load the IMDb dataset
dataset = datasets.load_dataset('stanfordnlp/imdb')

print("Dataset loaded successfully:")
print(dataset)

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [8]:
# Display an example from the training set
print("\nExample from training set:")
print(dataset['train'][1])


Example from training set:
{'text': '"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn\'t matter what one\'s political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn\'t true. I\'ve seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don\'t exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we\'re treated to the site of Vincent Gallo\'s throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display w

### Tokenization

We need to convert the text reviews into numerical tokens that the models can understand. We'll use a `AutoTokenizer` suitable for our chosen Transformer baseline (e.g., DistilBERT) and apply it to the entire dataset. For Mamba, a similar tokenization strategy will be used.

In [ ]:
# Initialize a tokenizer for the Transformer baseline (e.g., DistilBERT)
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
# TODO parametryize this with max_length
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512) # Default max_length, will be varied in scaling analysis

# Apply tokenization to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("\nTokenized dataset example:")
print(tokenized_dataset['train'][0])

In [ ]:
# Prepare data for training
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nDataset ready for training:")
print(tokenized_dataset['train'].column_names)

## 3. Approach 1: Transformer (Baseline)

In this section, we will implement and train a Transformer model. We'll start with fine-tuning a pre-trained model like DistilBERT, which is a good balance between performance and computational cost. We'll define the model, training arguments, and use the `Trainer` API for training.

### Fine-tuning DistilBERT

We will fine-tune a `distilbert-base-uncased` model for sequence classification. This involves loading the pre-trained model, configuring training arguments, and using the `Trainer` API to manage the training and evaluation process.

In [ ]:
# Define a custom callback for timing epochs
class TimingCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_end_time = time.time()
        epoch_duration = epoch_end_time - self.epoch_start_time
        print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

# Load pre-trained DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=1, # Changed from 10 to 1 for more frequent updates
    report_to="tensorboard" # Add this for loss tracking
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[TensorBoardCallback(), TimingCallback()] # Add TensorBoardCallback and TimingCallback
)

# Train the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print("\nDistilBERT Evaluation Results:")
print(results)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.233325,0.222963,0.913560,0.913342
2,0.200013,0.229751,0.930840,0.930832
3,0.145204,0.278247,0.932160,0.932160


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


DistilBERT Evaluation Results:
{'eval_loss': 0.2782467305660248, 'eval_accuracy': 0.93216, 'eval_f1': 0.932159789858165, 'eval_runtime': 378.56, 'eval_samples_per_second': 66.04, 'eval_steps_per_second': 4.129, 'epoch': 3.0}


## 4. Approach 2: Mamba

Here, we will implement and train a Mamba model. We can either fine-tune a pre-trained Mamba model (e.g., `state-spaces/mamba-130m` if available for classification) or build a small Mamba model from scratch and train it for sequence classification. This section will require custom model definition and potentially custom training loops if the `Trainer` API isn't directly compatible.

### Fine-tuning Mamba

For Mamba, we will define a custom `MambaForSequenceClassification` model, as `transformers` does not yet have a direct `AutoModelForSequenceClassification` for Mamba. We will initialize a Mamba block and add a classification head on top. The training process will then use the same `Trainer` API.

In [8]:
from transformers import AutoTokenizer
from transformers import GPTNeoXTokenizerFast

def prepare_mamba_dataset(dataset, max_length=1024):
    # 1. Initialize the Mamba tokenizer
    tokenizer = GPTNeoXTokenizerFast.from_pretrained("state-spaces/mamba-130m-hf")

    # 2. CRITICAL FOR MAMBA: Assign the EOS token as the padding token
    tokenizer.pad_token = tokenizer.eos_token

    # 3. Define the mapping tokenization function
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
            # We don't pad here; DataCollatorWithPadding will handle it dynamically per batch
        )

    # 4. Apply tokenization across the dataset splits
    print("Tokenizing dataset for Mamba...")
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    # 5. Format columns to match what the Trainer and PyTorch expect
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    if "label" in tokenized_dataset["train"].column_names:
        tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

    tokenized_dataset.set_format("torch")

    return tokenized_dataset, tokenizer

In [9]:
# ── Patch: mamba-ssm v2.x removed top-level ops that transformers still expects ──
import mamba_ssm

if not hasattr(mamba_ssm, 'selective_state_update'):
    from mamba_ssm.ops.selective_scan_interface import (
        selective_scan_fn,
        selective_state_update,
    )
    mamba_ssm.selective_state_update = selective_state_update
    mamba_ssm.selective_scan_fn     = selective_scan_fn

if not hasattr(mamba_ssm, 'mamba_inner_fn'):
    try:
        from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn
        mamba_ssm.mamba_inner_fn = mamba_inner_fn
    except ImportError:
        # v2 renamed / removed mamba_inner_fn; None makes transformers skip it
        mamba_ssm.mamba_inner_fn = None

print("mamba_ssm patched:",
      hasattr(mamba_ssm, 'selective_state_update'),
      hasattr(mamba_ssm, 'selective_scan_fn'),
      hasattr(mamba_ssm, 'mamba_inner_fn'))

mamba_ssm patched: True True True


In [14]:
class MambaForSequenceClassification(MambaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.backbone = MambaModel(config)  # must be 'backbone' to match checkpoint keys
        self.score = nn.Linear(config.hidden_size, self.num_labels, bias=False)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, use_cache=False)
        hidden  = outputs[0]  # (B, L, H)

        if attention_mask is not None:
            seq_lens = attention_mask.int().sum(-1) - 1
            pooled   = hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]
        else:
            pooled   = hidden[:, -1, :]

        logits = self.score(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states)

In [24]:
import time

from transformers import AutoTokenizer, TrainerCallback, TrainingArguments, Trainer, AutoConfig

class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

def fineTuneMambaClassification(tokenized_dataset, collator,batch_size=16):
    # 1. Initialize Tokenizer & handle padding token requirements
    tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
    tokenizer.pad_token = tokenizer.eos_token

    # 2. Initialize our custom classification model
    # Load the configuration with use_mamba_kernels set to False from the start
    config = AutoConfig.from_pretrained(
        "state-spaces/mamba-130m-hf",
        num_labels=2,
        use_mamba_kernels=True # Set this directly during config loading
    )

    model = MambaForSequenceClassification.from_pretrained(
        "state-spaces/mamba-130m-hf",
        config=config, # Pass the fully configured config
        ignore_mismatched_sizes=True
    )
    # Explicitly configure pad token ID to match the tokenizer configuration
    model.config.pad_token_id = tokenizer.pad_token_id


    # 3. Define Standard Training Arguments
    training_args = TrainingArguments(
        output_dir="./mamba_classification_results",
        eval_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        gradient_checkpointing=False,
        weight_decay=0.01,
        logging_steps=1,
        fp16=False,  # Recommended for custom CUDA compilation layouts
        bf16=True,  # Recommended for custom CUDA compilation layouts
        report_to="tensorboard"
    )

    # 4. Fire up the regular Hugging Face Trainer engine
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=collator,
        compute_metrics=compute_metrics, # Pass your accuracy/f1 metric helper
        callbacks=[TimingCallback()] # Add TensorBoardCallback and TimingCallback
    )

    trainer.train()

In [22]:
import sys
import torch # Import torch to get its __file__ attribute
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ # Point to an actual file from an imported module

# 1. Process data for Mamba (supporting 1024 text length safely!)
tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=128)

# 2. Build the collator with the explicit Mamba tokenizer reference
data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

# 3. Pass tokenized_dataset and data_collator into your trainer!
fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 15212.70it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.068800,0.268539,0.891960,0.891842
2,0.006853,0.353905,0.889000,0.888807
3,0.000058,0.605186,0.895920,0.895914
4,0.000547,0.820401,0.892800,0.892767
5,0.000001,0.867474,0.894560,0.894550


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]


In [25]:
import sys
import torch # Import torch to get its __file__ attribute
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ # Point to an actual file from an imported module

# 1. Process data for Mamba (supporting 1024 text length safely!)
tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=256)

# 2. Build the collator with the explicit Mamba tokenizer reference
data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

# 3. Pass tokenized_dataset and data_collator into your trainer!
fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 14609.67it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.123468,0.201960,0.922920,0.922892
2,0.005053,0.254438,0.924240,0.924193
3,0.000107,0.395591,0.925760,0.925757
4,0.000044,0.534691,0.927400,0.927397
5,0.000035,0.573302,0.927600,0.927599


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]


Epoch 1 completed in 253.29 seconds


...

Epoch 3 completed in 253.70 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]


Epoch 4 completed in 253.79 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]


Epoch 5 completed in 255.26 seconds


In [26]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=512)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 17198.80it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.029462,0.168756,0.940000,0.939974
2,0.019181,0.182540,0.944120,0.944112
3,0.000020,0.325445,0.942840,0.942834
4,0.000719,0.412771,0.945440,0.945440
5,0.000001,0.441448,0.944880,0.944879


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]


Epoch 1 completed in 445.90 seconds


...

Epoch 3 completed in 450.96 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


Epoch 4 completed in 444.77 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


Epoch 5 completed in 446.99 seconds


In [ ]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=1024)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

## 5. Approach 3: Scaling Analysis

This section will focus on comparing both architectures across different sequence lengths (128, 512, 1024 tokens). We will measure:
-   **Training time per epoch**
-   **Inference time**
-   **Quality metrics**: Accuracy, F1-score

This will involve re-tokenizing the dataset with different `max_length` values and repeating the training and evaluation steps for each model and sequence length.

In [ ]:
# Helper function to compute metrics
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}